# 05 — NepBERTa Demo (single-input prediction)

A minimal notebook that:

1. Loads the saved fine-tuned NepBERTa model **once**.
2. Takes a Nepali sentence (edit `sample_text`) and prints:
   - predicted label,
   - confidence (softmax probability),
   - per-class probabilities,
   - wall-clock inference time.

No training happens here. If the model directory is missing, run
`python -m src.run_nepberta_local` (or the Colab notebook 03) first.

Saved model layout (after Phase 3):
```
outputs/models/nepberta_finetuned/
├── config.json
├── model.safetensors
├── tokenizer.json
├── tokenizer_config.json
├── special_tokens_map.json
└── vocab.txt
```


In [ ]:
# Auto-reload edits in src/ without restarting the kernel.
%load_ext autoreload
%autoreload 2


In [ ]:
# ============================================================================
# IMPORTS + PATH FIX + DEVICE DETECTION
# ============================================================================
import sys, time
from pathlib import Path

# Make `from src import ...` work whether the notebook starts in
# `notebooks/` or at the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import pandas as pd

from src import config
from src.models import nepberta as nb

# Pick the fastest available backend. NepBERTa inference works on all three;
# CPU is just slower (~1-2 seconds per sentence vs ~50 ms on GPU).
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'Using device: {DEVICE}')


## Load NepBERTa once

Loading is the slow part (~5-10 seconds — a 420 MB safetensors file plus
tokenizer). After this cell runs the model object is cached in the kernel
and every subsequent prediction is fast.


In [ ]:
# `nb.load_model()` reads from config.NEPBERTA_MODEL_DIR by default.
# It returns (tokenizer, model). We then move the model to the chosen
# device and put it in eval() mode (disables dropout — important for
# deterministic, deployment-style predictions).
tokenizer, model = nb.load_model()
model = model.to(DEVICE)
model.eval()
print(f'NepBERTa loaded — {sum(p.numel() for p in model.parameters()):,} parameters')


## Predict

Edit the `sample_text` string below and re-run the cell. The output
shows the predicted label in bold, confidence as a percentage, the
full per-class probability breakdown, and the wall-clock inference
time for that single prediction.


In [ ]:
# ============================================================================
# EDIT THIS LINE — then re-run the cell
# ============================================================================
sample_text = 'मलाई यो काम एकदमै मन पर्‍यो'

# ----- Predict + time -------------------------------------------------------
# `time.perf_counter()` is the highest-precision wall clock in Python — use
# this for benchmarking, not `time.time()` (which can jump on system clock
# adjustments).
t0       = time.perf_counter()
pred_id  = nb.predict(model, tokenizer, [sample_text])[0]
probs    = nb.predict_proba(model, tokenizer, [sample_text])[0]
elapsed_ms = (time.perf_counter() - t0) * 1000

# ----- Display --------------------------------------------------------------
pred_label = config.LABEL_NAMES[pred_id]
confidence = probs[pred_id]

print(f'Input:       {sample_text}')
print(f'Prediction:  {pred_label}  (confidence {confidence:.2%})')
print(f'Inference:   {elapsed_ms:.1f} ms')
print()

# Per-class probability breakdown as a tidy single-row DataFrame.
pd.DataFrame({
    name: [f'{probs[i]:.4f}']
    for i, name in enumerate(config.LABEL_NAMES)
}, index=['probability'])


## Try-it-yourself examples

Swap in different sentences to see how confidence and per-class
probabilities shift:

```python
# Clear positive
sample_text = 'मलाई यो काम एकदमै मन पर्‍यो'

# Clear negative
sample_text = 'सरकार भ्रष्ट छ, देशको अवस्था बिग्रिएको छ'

# Neutral / news-like
sample_text = 'काठमाडौंमा आज पानी पर्‍यो'

# Negation — flip-test
sample_text = 'यो फिल्म राम्रो छैन'

# Short ambiguous — watch the confidence drop
sample_text = 'ठिक छ'
```
